<a href="https://colab.research.google.com/github/fairclj/HMI-2026-Demo/blob/main/Sample_Machine_Learning_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Sample Python Notebook - Diabetes Prediction (NIH-NIDDK Dataset)**  
##### **Notebook Creator: Jamie Fairclough, PhD, MPH, MSPharm (Dartmouth)**  
------------------------------------------------------------

**Dataset Source:** https://www.kaggle.com/uciml/pima-indians-diabetes-database

**Description:** *"This dataset is originally from the National Institute of Diabetes and Digestive and Kidney Diseases. The objective of the dataset is to diagnostically predict whether or not a patient has diabetes, based on certain diagnostic measurements included in the dataset. Several constraints were placed on the selection of these instances from a larger database. In particular, all patients here are females at least 21 years old of Pima Indian heritage."*  
  
**Note: This dataset is being used for demonstration purposes only.**

In [ ]:

# @title **Import necessary Python libraries**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
import pandas as pd
import numpy as np
import io
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import scipy.stats as stats
from scipy.stats import skew, norm, probplot, boxcox, f_oneway

import warnings
warnings.filterwarnings("ignore")

print("Python libraries successfully imported.")

In [ ]:
# @title **Upload dataset (CSV format)**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
# Upload file
uploaded = files.upload()

# Get uploaded filename automatically
filename = next(iter(uploaded))

# Read into dataframe
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print("Data successfully uploaded")

In [ ]:
# @title **View first 10 rows of data**

df.head(10) # CHANGE NUMBER TO INCREASE OR DECREASE THE NUMBER OF ROWS

In [ ]:
# @title **View a list of all variables**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
attribute_list = list(df.columns)
print("Variables in the dataset:")
print("")
print(attribute_list)

In [ ]:
# @title **View the number of rows and columns in the dataset**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
print("Number of rows and columns:")
print("")
df.shape   # (rows, columns)

In [ ]:
# @title **View general information about the dataset**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
df.info()

In [ ]:
# @title **Describe the data (and data type) for each variable**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#

# Describe all columns
summary = df.describe(include='all').T

# Replace NaN values with 'NA'
summary = summary.fillna('NA')

# Insert formatted variable name + data type
summary.insert(
    0,
    'Variable',
    [f"<b>{col}</b><br><span style='font-weight:normal; font-size:smaller;'>[{df[col].dtype}]</span>" for col in summary.index]
)

# Reset index to clean up formatting
summary.reset_index(drop=True, inplace=True)

# Format numeric columns to 3 decimal places only
styled_summary = (
    summary.style
    .format(precision=3, na_rep="NA")  # Format numbers to 3 decimals
    .set_table_styles([
        {'selector': 'th', 'props': [('font-weight', 'bold'), ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'center')]}
    ])
    .hide(axis='index')  # Remove default row index
)

styled_summary

In [ ]:
# @title **Replace missing values or impossible values (e.g., zero) with the median**

# Define a list of variables where 0 values should be replaced with the median
variables_to_replace_zeros = ['Glucose', 'D_BP', 'Skin_Thickness', 'Insulin', 'BMI']  # You can modify this list with your selected variables

# Replace missing (NaN) values with the median for all columns
df = df.apply(lambda x: x.fillna(x.median()) if x.isnull().any() else x)

# Replace 0 values with the median for selected variables
for var in variables_to_replace_zeros:
    if var in df.columns:
        median_value = df[var].replace(0, pd.NA).median()  # Find the median excluding 0s
        df[var] = df[var].replace(0, median_value)  # Replace 0 with the median

# Output the result (optional)
print("Missing and 0 values replaced with median where applicable.")

In [ ]:
# @title **Description of just one variable**

# Define the variable name (this can be dynamic)
variable = 'BMI'  # CHANGE VARIABLE HERE


#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#

# Print the title dynamically
print(f"Quantitative Description of {variable}:")
print("")

# Get the description and round it to 3 decimal places
description = df[variable].describe().round(3)

# Print the rounded description
print(description)

In [ ]:
# @title **Number of responses and missing values for each variable**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#

# Count non-null and missing responses
response_counts = df.count()
missing_counts = df.isnull().sum()

# Combine into one DataFrame
df_count = pd.DataFrame({
    'Variable': response_counts.index,
    '# of Responses': response_counts.values,
    '# Missing': missing_counts.values
})

# Display as styled table: bold headers and center-aligned values
df_count.style.set_table_styles([
    {'selector': 'th', 'props': [('font-weight', 'bold'), ('text-align', 'center')]},
    {'selector': 'td', 'props': [('text-align', 'center')]}
])

In [ ]:
# @title **View 15 random rows of data**
np.random.seed()
df.sample(n=15) # CHANGE NUMBER HERE TO INCREASE OR DECREASE NUMBER OF RANDOM ROWS

In [ ]:
# @title **Change a variable's data type (e.g., from integer to category)**

# Choose variable to change
variable_to_change = 'Outcome' # CHANGE THIS VARIABLE
new_data_type = 'category' # CHANGE THIS TO THE NEW DATA TYPE (options include category, int64, float)

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
# Capture original data type
original_dtype = df[variable_to_change].dtype

# Change data type (options include category, float, int64)
df[variable_to_change] = df[variable_to_change].astype(new_data_type)

# Capture new data type
new_dtype = df[variable_to_change].dtype

# Print result dynamically
print(f"'{variable_to_change}' variable changed from {original_dtype} to {new_dtype}")
print("")
print("Note: You can revise the code to change the data type for a different variable.")

## **Exploratory Data Analysis**

In [ ]:
# @title **Pearson's correlation table**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
df.corr()

In [ ]:
# @title **Pearson's correlation heatmap**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
plt.figure(figsize=(10,6))
sns.set(font_scale=0.8)
plt.rcParams["axes.labelsize"] = 0.5
sns.heatmap(df.corr(), annot=True);

In [ ]:
# @title **Pearson's correlation heatmap in 'coolwarm' color**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
plt.figure(figsize=(10, 6))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm");

In [ ]:
# @title **Spearman's Rank correlation table**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
df.corr(method='spearman')

In [ ]:
# @title **Spearman's Rank correlation heatmap**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
plt.figure(figsize=(10,6))
sns.set(font_scale=0.8)
plt.rcParams["axes.labelsize"] = 0.5
sns.heatmap(df.corr(method='spearman'), annot=True, cmap="YlGnBu");

In [ ]:
# @title **Phi-K correlation heatmap**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#

import os
import sys
import subprocess

# Suppress pip install output
subprocess.run([sys.executable, "-m", "pip", "install", "phik"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import phik
from phik import resources, report
from phik.report import plot_correlation_matrix
from phik import phik_matrix

# Identify interval (numerical) columns
interval_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Create correlation map using Phi-K
plt.figure(figsize=(10, 6))
sns.set(font_scale=0.8)
plt.rcParams["axes.labelsize"] = 0.5

# Compute and plot Phi-K correlation matrix with interval columns specified
phik_corr = df.phik_matrix(interval_cols=interval_cols)
sns.heatmap(phik_corr, annot=True, cmap="coolwarm")

plt.title("Phi_K Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# @title **Example of a histogram**
plt.hist(df['BMI'], color='blue'); # CHANGE VARIABLE HERE

In [ ]:
# @title **Example of a distribution plot**
sns.distplot(df['BMI'], color='red', rug=True); # CHANGE VARIABLE AND COLOR HERE

In [ ]:
# @title **Example of a violin plot**
sns.violinplot(df['Age'], color='yellow', orient='h'); # CHANGE VARIABLE AND COLOR HERE

In [ ]:
# @title **Example of categorical plot**
# Define the plot variables
x_var = "Outcome"
y_var = "Glucose"

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#

# Create the plot
g = sns.catplot(x=x_var, y=y_var, data=df, kind='boxen', height=6, aspect=1.6, estimator=np.mean);

# Dynamically set axis labels
g.set_axis_labels(x_var, y_var)
g.set_titles("{col_name}")

# Optionally set font size
g.set(xlabel=x_var, ylabel=y_var)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

In [ ]:
# @title **Define a function to create a boxplot and histogram combo**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
def histogram_boxplot(feature, figsize=(10,6), bins=None):
    """ Boxplot and histogram combined (horizontal boxplot version)
    feature: 1-d feature array
    figsize: size of fig (default (10,6))
    bins: number of bins (default None / auto)
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    from scipy.stats import norm

    sns.set(font_scale=1)
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2,
        sharex=True,
        gridspec_kw={"height_ratios": (.25, .75)},
        figsize=figsize
    )

    # Horizontal boxplot
    sns.boxplot(x=feature, ax=ax_box2, showmeans=True, color='red')

    # Histogram
    if bins:
        sns.histplot(feature, kde=False, ax=ax_hist2, bins=bins)
    else:
        sns.histplot(feature, kde=False, ax=ax_hist2)

    # Add central tendency lines to histogram
    ax_hist2.axvline(np.mean(feature), color='black', linestyle='--', label='Mean')
    ax_hist2.axvline(np.median(feature), color='green', linestyle='-', label='Median')
    ax_hist2.axvline(feature.mode()[0], color='red', linestyle='dashed', linewidth=1, label='Mode')

    ax_hist2.legend()
    plt.tight_layout()
    plt.show()

print("Function successfully created")

In [ ]:
# @title **Example of a combination boxplot and histogram**

histogram_boxplot(df.BMI) # CHANGE THE CONTINUOUS VARIABLE HERE

In [ ]:
# @title **Example of boxplot with the primary outcome and a continuous variable**

# Define variables
x_var = "Age" # CHANGE VARIABLE HERE
y_var = "Outcome"  # CHANGE VARIABLE HERE

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#

# Plot
plt.figure(figsize=(10, 5))
sns.boxplot(x=x_var, y=y_var, data=df)

# Dynamic labels and title
plt.xlabel(x_var, fontsize=12)
plt.ylabel(y_var, fontsize=12)
plt.title(f'Boxplot for {y_var} vs. {x_var}', fontsize=14)

plt.show()

In [ ]:
# @title **Boxplots for all continuous variables (to view outliers)**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
plt.figure(figsize=(20,30))

for i, variable in enumerate(df):
                     plt.subplot(5,4,i+1)
                     plt.boxplot(df[variable],whis=1.5)
                     plt.tight_layout()
                     plt.title(variable)

plt.show()

In [ ]:
# @title **Distributions of all continuous variables**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
# from scipy.stats import norm
all_col = df.select_dtypes(include=np.number).columns.tolist()
plt.figure(figsize=(17,75))

for i in range(len(all_col)):
    plt.subplot(18,3,i+1)
    plt.hist(df[all_col[i]])
    plt.tight_layout()
    plt.title(all_col[i],fontsize=25)

plt.show()

In [ ]:
# @title **Example of a scatterplot**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#

# Create the scatterplot
sns.set(style="white")  # Clean background without gridlines
plt.figure(figsize=(10, 6))

# Scatterplot
sns.scatterplot(x='BMI', y='Glucose', hue='Outcome', data=df, palette='Set2')

# Add title and axis labels
plt.title('BMI vs Glucose by Outcome', fontsize=16)
plt.xlabel('BMI', fontsize=14)
plt.ylabel('Glucose', fontsize=14)

# Remove gridlines
plt.grid(False)

# Adjust layout to prevent labels from being cut off
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# @title **Example of a joint plot (3 variables)**
g = sns.jointplot(
    x='BMI', # CHANGE VARIABLE HERE
    y='Age', # CHANGE VARIABLE HERE
    hue='Outcome',
    data=df,
    palette='Set2',
    height=6,
)

# Set x-axis and y-axis limits
g.ax_joint.set_xlim(left=0)
g.ax_joint.set_ylim(bottom=0)

# Set y-axis ticks to increase by 10
y_min, y_max = g.ax_joint.get_ylim()
g.ax_joint.set_yticks(np.arange(10, y_max + 10, 10))

# Add title
plt.suptitle("Joint Plot of BMI vs. Age by Outcome", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# @title **Example of a joint plot of a different kind (2 variables)**

sns.set(font_scale=0.8)
plt.rcParams["axes.labelsize"] = 0.5

sns.jointplot(
    x='BMI', # CHANGE VARIABLE HERE
    y='Age', # CHANGE VARIABLE HERE #---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
    kind='hex',
    data=df,
    palette='Set2',
    height=6  # Smaller and more compact
);

In [ ]:
# @title **Another example of a joint plot of a different kind (2 variables)**
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(font_scale=0.8)
plt.rcParams["axes.labelsize"] = 0.5

sns.jointplot(
    x='BMI', # CHANGE VARIABLE HERE
    y='Age', # CHANGE VARIABLE HERE #---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
    kind='kde',
    shade = True,
    data=df,
    palette='Set2',
    height=6  # Smaller and more compact
);

In [ ]:
# @title **Example of a time series plot**

# Define x and y columns
x_col = 'Age' # CHANGE VARIABLE HERE
y_col = 'Pregnancies' # CHANGE VARIABLE HERE

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#

# Set up figure and axis
fig, ax = plt.subplots(figsize=(10, 6))

# Create lineplot
sns.lineplot(x=x_col, y=y_col, data=df, ax=ax)

# Set axis labels and title explicitly
ax.set_xlabel(x_col, fontsize=12)
ax.set_ylabel(y_col, fontsize=12)
ax.set_title(f'{y_col} by {x_col}', fontsize=14)

# Ensure labels are not cut off
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# @title **Example of a strip plot**

# Define axis variables
x_var = "Outcome"  # CHANGE VARIABLE HERE
y_var = "Age"      # CHANGE VARIABLE HERE

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
sns.stripplot(data=df, x=x_var, y=y_var, jitter=True)
plt.xlabel(x_var, fontsize=12)
plt.ylabel(y_var, fontsize=12)
plt.title(f'{y_var} by {x_var}', fontsize=14)
plt.show()

In [ ]:
# @title **Example of a swarm plot**

# Define axis variables
x_var = "Outcome"  # CHANGE VARIABLE HERE
y_var = "Age"      # CHANGE VARIABLE HERE

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
sns.swarmplot(data=df, x=x_var, y=y_var)
plt.xlabel(x_var, fontsize=12)
plt.ylabel(y_var, fontsize=12)
plt.title(f'{y_var} by {x_var}', fontsize=14)
plt.show()

**Link to other data visualizations: https://seaborn.pydata.org/examples/index.html**

## **Inferential Statistics**

In [ ]:
# @title **Import additional inferential statistics libraries**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
import math
from scipy.stats import ttest_1samp
from scipy.stats import ttest_ind

print("Libraries successfully imported.")

In [ ]:
# @title **Example of one-sample t-test***

# One-sample t-test
variable = 'BMI'  # CHANGE VARIABLE HERE
threshold = 30       # CHANGE THRESHOLD HERE

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
t_statistic, p_value = ttest_1samp(df[variable], threshold)

# Significance level
alpha = 0.05  # You can change this alpha to 0.01 or another value

# Format p-value
formatted_p = "p < 0.001" if p_value < 0.001 else f"{p_value:.3f}"

# Decision
reject_null = p_value < alpha
decision = "Reject the null hypothesis." if reject_null else "Fail to reject the null hypothesis."

# Interpretation
if reject_null:
    if t_statistic > 0:
        interpretation = f"The average {variable} (for all patients) is significantly higher than {threshold}."
    else:
        interpretation = f"The average {variable} (for all patients) is significantly lower than {threshold}."
else:
    interpretation = f"The average {variable} (for all patients) is not significantly different from {threshold}."

# Output
print(f"Null Hypothesis: The average {variable} is equal to {threshold}.")
print("")
print(f"One sample t-test results:\n t-statistic = {t_statistic:.3f}\n p-value = {formatted_p}")
print(f" Decision: {decision}")
print(f" Interpretation: {interpretation}")

In [ ]:
# @title **Example of independent samples t-test (specific to this dataset)**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#

# Define the variable and group column
variable = 'BMI'
group_col = 'Outcome'

# Define readable group labels
label_map = {0: "Non-Diabetics", 1: "Diabetics"}

# Identify unique groups
groups = df[group_col].unique()
group1_label, group2_label = groups[0], groups[1]

# Extract the data for each group
group1 = df[variable][df[group_col] == group1_label]
group2 = df[variable][df[group_col] == group2_label]

# Perform the t-test
t_statistic, p_value = ttest_ind(group1, group2)

# Significance level
alpha = 0.05

# Format p-value
formatted_p = "p < 0.001" if p_value < 0.001 else f"{p_value:.3f}"

# Decision
reject_null = p_value < alpha
decision = "Reject the null hypothesis." if reject_null else "Fail to reject the null hypothesis."

# Interpretation using label_map
group1_name = label_map.get(group1_label, str(group1_label))
group2_name = label_map.get(group2_label, str(group2_label))

if reject_null:
    if t_statistic > 0:
        interpretation = f"The average {variable} for {group1_name} is significantly higher than for {group2_name}."
    else:
        interpretation = f"The average {variable} for {group1_name} is significantly lower than for {group2_name}."
else:
    interpretation = f"The average {variable} for {group1_name} is not significantly different from {group2_name}."

# Output
print(f"Null Hypothesis: The average {variable} for {group1_name} is equal to the average {variable} for {group2_name}.")
print("")
print(f"Independent samples t-test results:\n t-statistic = {t_statistic:.3f}\n p-value = {formatted_p}")
print(f" Decision: {decision}")
print(f" Interpretation: {interpretation}")

In [ ]:

# @title **ANOVA (specific to this dataset)**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#

import statsmodels.api as sm
from statsmodels.formula.api import ols

# Define the variable and group column
variable = 'Glucose'
group_col = 'Outcome'

# Optional: mapping group labels to readable names
label_map = {0: 'Non-Diabetics', 1: 'Diabetics'}

# Create formula dynamically
formula = f'{variable} ~ C({group_col})'

# Fit the model
model = ols(formula, data=df).fit()

# Perform ANOVA
anova_table = sm.stats.anova_lm(model, typ=2)

# Extract test results
f_stat = anova_table['F'][0]
p_value = anova_table['PR(>F)'][0]

# Significance level
alpha = 0.05

# Format p-value
formatted_p = "p < 0.001" if p_value < 0.001 else f"{p_value:.3f}"

# Decision
reject_null = p_value < alpha
decision = "Reject the null hypothesis." if reject_null else "Fail to reject the null hypothesis."

# Get group labels for interpretation
groups = df[group_col].unique()
group_names = [label_map.get(g, str(g)) for g in groups]

# Interpretation
if reject_null:
    interpretation = f"There is a statistically significant difference in {variable} across groups: {', '.join(group_names)}."
else:
    interpretation = f"There is no statistically significant difference in {variable} across groups: {', '.join(group_names)}."

# Output
print(f"Null Hypothesis: The average {variable} is the same across all {group_col} groups.\n")
print(f"ANOVA results:\n F-statistic = {f_stat:.3f}\n p-value = {formatted_p}")
print(f" Decision: {decision}")
print(f" Interpretation: {interpretation}")

In [ ]:
# @title **Chi-square test (specific to this dataset)**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
from scipy.stats import chisquare
crosstab = pd.crosstab(df['Outcome'], df['Pregnancies']>10)
crosstab

In [ ]:
# @title **Map outcome labels back to values -- (only if needed for machine learning)**
#df.Outcome=df['Outcome'].map({'No Diabetes':0, 'Diabetes':1})
#print("Code successful")

# Check to make sure outcome displays values again.**
#df.Outcome.head(10)

## **Supervised Machine Learning (Binary Classification)**

**Two examples:**
- Logistic Regression  
- Decision Tree  

**LOGISTIC REGRESSION:** Additional Info on Logistic Regression: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

## **Logistic Regression**

In [ ]:
# @title **Split data between training and test set**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#

from sklearn.model_selection import train_test_split

X = df.drop('Outcome',axis=1) # drop the outcome
y = df['Outcome'] # pick the outcome

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size = 0.3, random_state = 42)

print("Data successfully split")

In [ ]:
# @title **Create confusion matrix to depict true and false positives/negatives**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
from sklearn.metrics import classification_report,confusion_matrix


def make_confusion_matrix(y_actual,y_predict,labels=[0, 1]):
    '''
    y_predict: class prediction
    y_actual: actual truth
    '''
    cm=confusion_matrix( y_predict,y_actual, labels=[0, 1])
    df_cm = pd.DataFrame(cm, index = [i for i in ["0","1"]],
                  columns = [i for i in ['0','1']])
    group_counts = ["{0:0.0f}".format(value) for value in
                cm.flatten()]
    group_percentages = ["{0:.2%}".format(value) for value in
                         cm.flatten()/np.sum(cm)]
    labels = [f"{v1}\n{v2}" for v1, v2 in
              zip(group_counts,group_percentages)]
    labels = np.asarray(labels).reshape(2,2)
    plt.figure(figsize = (7,5))
    sns.heatmap(df_cm, annot=labels,fmt='')
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

print("Function successfully created")

In [ ]:
# @title **Build the logistic regression model**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
from sklearn.linear_model import LogisticRegression

# Fit the model on training data
model = LogisticRegression(solver="liblinear")
model.fit(X_train, y_train)

# Predict on test data
y_predict = model.predict(X_test)

# Suppress coefficient output (commented out)
# coef_df = pd.DataFrame(model.coef_)
# coef_df['intercept'] = model.intercept_
# print(coef_df)  # Logistic regression coefficients are difficult to interpret, so this output is suppressed

# Get model accuracy
model_score = model.score(X_test, y_test)

# Print model score alone
print(model_score)

# Interpret model accuracy in plain English
accuracy_pct = round(model_score * 100, 2)
inaccuracy_pct = round((1 - model_score) * 100, 2)

print(f"Logistic regression model was accurate {accuracy_pct}% of the time and inaccurate {inaccuracy_pct}% of the time.")

In [ ]:
# @title **AUC-ROC curve for logistic regression to find optimal threshold**

from sklearn.metrics import roc_auc_score, roc_curve

# Compute ROC AUC
logit_roc_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
fpr, tpr, thresholds = roc_curve(y_test, model.predict_proba(X_test)[:, 1])

# Plot
plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, label=f'ROC curve (area = {logit_roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')  # Diagonal line
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc='lower right')
plt.show()

In [ ]:
# @title **Optimal cutoff where true positive rate is high and false positive rate is low**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
model = LogisticRegression(solver="liblinear")
model.fit(X_train, y_train)

fpr, tpr, thresholds = roc_curve(y_test, model.predict_proba(X_test)[:, 1])

optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
print(f"Optimal threshold: {optimal_threshold:.3f}")

In [ ]:
# @title **Apply optimal threshold value to the logistic regression model**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
target_names = ['0', '1']

y_pred_tr = (model.predict_proba(X_train)[:, 1] > optimal_threshold).astype(int)
y_pred_ts = (model.predict_proba(X_test)[:, 1] > optimal_threshold).astype(int)

print("Optimal threshold value applied")

In [ ]:
# @title **Make confusion matrix**
make_confusion_matrix(y_test,y_pred_ts)

In [ ]:
# @title **Compare accuracy scores**
from sklearn.metrics import accuracy_score

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
print('Accuracy on train data:',accuracy_score(y_train, y_pred_tr) )
print('Accuracy on test data:',accuracy_score(y_test, y_pred_ts))

## **Decision Tree**

**DECISION TREE: https://scikit-learn.org/stable/modules/tree.html#**

In [ ]:
# @title **Import libraries for additional models**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
# Libraries for different classifiers
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import tree

# Libraries for model tuning and evaluation metrics
from sklearn import metrics
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV

print("Additional libraries imported successfully")

In [ ]:
# @title **Create a copy of dataframe**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
data=df
print("Code successful")

In [ ]:
# @title **Separate the outcome from the other variables**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
X = data.drop('Outcome',axis=1)
y = data['Outcome']
print("Code successful")

In [ ]:
# @title **Train the model**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1,stratify=y)
print(X_train.shape, X_test.shape)

In [ ]:
# @title **Build decision tree model**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#

dtree = DecisionTreeClassifier(class_weight={0:0.35,1:0.65},random_state=1)

# Parameters:
parameters = {'max_depth': np.arange(2,10),
              'min_samples_leaf': [5, 7, 10, 15],
              'max_leaf_nodes' : [2, 3, 5, 10,15],
              'min_impurity_decrease': [0.0001,0.001,0.01,0.1]
             }

# Scoring method used to compare parameter combinations
scorer = metrics.make_scorer(metrics.recall_score)

# Run the grid search
grid_obj = GridSearchCV(dtree, parameters, scoring=scorer,n_jobs=-1)
grid_obj = grid_obj.fit(X_train, y_train)

# Set the clf to the best combination of parameters
dtree = grid_obj.best_estimator_

# Fit the best algorithm to the data.
dtree.fit(X_train, y_train)

print("Decision tree built")

In [ ]:
# @title **Function to calculate metric scores (Accuracy, Recall, Precision, F1)**
##  Function to calculate different metric scores of the model - Accuracy, Recall, Precision, and F1 Scores

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
from sklearn import metrics  # Ensure this is imported

def get_metrics_score(model, flag=True):
    '''
    model : classifier to predict values of X
    '''
    score_list = []

    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)

    train_recall = metrics.recall_score(y_train, pred_train)
    test_recall = metrics.recall_score(y_test, pred_test)

    train_precision = metrics.precision_score(y_train, pred_train)
    test_precision = metrics.precision_score(y_test, pred_test)

    # NEW: Compute F1 scores
    train_f1 = metrics.f1_score(y_train, pred_train)
    test_f1 = metrics.f1_score(y_test, pred_test)

    # Append all scores to the list
    score_list.extend((train_acc, test_acc, train_recall, test_recall, train_precision, test_precision, train_f1, test_f1))

    if flag == True:
        print("Accuracy on training set  :", train_acc)
        print("Accuracy on test set      :", test_acc)
        print("Recall on training set    :", train_recall)
        print("Recall on test set        :", test_recall)
        print("Precision on training set :", train_precision)
        print("Precision on test set     :", test_precision)
        print("F1 Score on training set  :", train_f1)
        print("F1 Score on test set      :", test_f1)

    return score_list

print("Code successful.")

In [ ]:
# @title **Function to make confusion matrix**

#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#
def make_confusion_matrix(model,y_actual,labels=[1, 0]):

    y_predict = model.predict(X_test)
    cm=metrics.confusion_matrix( y_actual, y_predict, labels=[0, 1])
    df_cm = pd.DataFrame(cm, index = [i for i in ["Actual - No","Actual - Yes"]],
                  columns = [i for i in ['Predicted - No','Predicted - Yes']])
    group_counts = ["{0:0.0f}".format(value) for value in
                cm.flatten()]
    group_percentages = ["{0:.2%}".format(value) for value in
                         cm.flatten()/np.sum(cm)]
    labels = [f"{v1}\n{v2}" for v1, v2 in
              zip(group_counts,group_percentages)]
    labels = np.asarray(labels).reshape(2,2)
    plt.figure(figsize = (10,7))
    sns.heatmap(df_cm, annot=labels,fmt='')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')

    print("Code successful")

print("Code successful.")

In [ ]:
# @title **Fit model and visualize confusion matrix**
#---------------------DO NOT CHANGE ANY CODE BELOW THIS LINE---------------------------------------------#

# Fit the model
dtree = DecisionTreeClassifier(random_state=1)
dtree.fit(X_train, y_train)

# Calculate metrics (accuracy, recall, precision, and F1 scores)
scores = get_metrics_score(dtree)

# Unpack the scores (optional for direct access)
train_acc, test_acc, train_recall, test_recall, train_precision, test_precision, train_f1, test_f1 = scores

# Print F1 scores explicitly
print(f"F1 Score on training set: {train_f1:.2f}")
print(f"F1 Score on test set:     {test_f1:.2f}")

# Create a confusion matrix
make_confusion_matrix(dtree, y_test)

In [ ]:
# @title **Fit model for decision tree classifier**

from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(criterion='gini',class_weight={0:0.15,1:0.85},random_state=1)
model.fit(X_train, y_train)

In [ ]:
# @title **Function to calculate recall score**
def get_recall_score(model):
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    print("Recall on training set : ",metrics.recall_score(y_train,pred_train))
    print("Recall on test set : ",metrics.recall_score(y_test,pred_test))

print("Code successful.")

In [ ]:
# @title **Show decision tree**

# If using pandas DataFrame for X_train:
feature_names = X_train.columns

# Plot decision tree
import matplotlib.pyplot as plt
from sklearn import tree

plt.figure(figsize=(20, 30))
out = tree.plot_tree(model,
                     feature_names=feature_names,
                     filled=True,
                     fontsize=9,
                     node_ids=False,
                     class_names=None)

for o in out:
    pass  # to suppress output if needed

In [ ]:
# @title **List most important features based on decision tree**

print (pd.DataFrame(model.feature_importances_, columns = ["Imp"], index = X_train.columns).sort_values(by = 'Imp', ascending = False))

In [ ]:
# @title **Visualize feature importance graph**
feature_names = X_train.columns
importances = dtree.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(10,8))
plt.title('Feature Importance')
plt.barh(range(len(indices)), importances[indices], color='blue', align='center')
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

#---------------------Interpret in Words---------------------------------------------#
# Get top 3 most important features
sorted_idx = np.argsort(importances)[::-1]
top_features = feature_names[sorted_idx[:3]].tolist()

# Print interpretation
print(f"\nBased on the decision tree, the most important feature was {top_features[0]} followed by {top_features[1]}.")
print(f"If time and/or resources are limited, focus efforts on addressing {top_features[0]} and {top_features[1]}.")